# Shopee Malaysia Google Play Reviews Preprocessing

This notebook covers Member 1 responsibilities only: data acquisition, cleaning, NLP preprocessing, basic EDA, dataset validation, and final export.

## 1. Setup and Package Verification

The Shopee Malaysia Google Play package is `com.shopee.my`. It is verified from the Google Play app details URL: `https://play.google.com/store/apps/details?id=com.shopee.my`.

In [1]:
from collections import Counter
import html
import re
import string
from pathlib import Path

import nltk
import pandas as pd
from google_play_scraper import Sort, reviews
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

APP_PACKAGE = "com.shopee.my"
TARGET_REVIEWS = 10_000
BATCH_SIZE = 200
PROGRESS_INTERVAL = 500

print(f"Google Play package name: {APP_PACKAGE}")
print("Verified using: https://play.google.com/store/apps/details?id=com.shopee.my")

Google Play package name: com.shopee.my
Verified using: https://play.google.com/store/apps/details?id=com.shopee.my


## 2. Shared Constants

These constants are used across the data collection, cleaning, NLP, EDA, and validation steps.

In [2]:
RAW_COLUMNS = [
    "review_id",
    "rating",
    "review_date",
    "app_version",
    "thumbs_up_count",
    "original_text",
]

FINAL_COLUMNS = RAW_COLUMNS + ["cleaned_text"]

SENTIMENT_WORDS_TO_KEEP = {
    "good",
    "bad",
    "excellent",
    "terrible",
    "love",
    "hate",
    "best",
    "worst",
}

MALAY_STOPWORDS = {
    "ada", "adalah", "akan", "aku", "anda", "apa", "atau", "bagaimana",
    "bagi", "bahawa", "banyak", "baru", "belum", "boleh", "buat", "dalam",
    "dan", "dapat", "dari", "daripada", "dengan", "dia", "di", "ini",
    "itu", "jadi", "jika", "juga", "kami", "kamu", "kan", "ke", "kerana",
    "lagi", "lah", "lebih", "mereka", "nak", "ni", "nya", "oleh", "pada",
    "pun", "saja", "saya", "sebagai", "semua", "sini", "sudah", "tak",
    "telah", "tentang", "tersebut", "tetapi", "tidak", "untuk", "yang",
}

## 3. Data Collection

Reviews are collected with `google-play-scraper`, using continuation tokens until 10,000 reviews are collected or Google Play returns no more reviews.

In [3]:
def scrape_google_play_reviews(app_package, target_reviews):
    """Collect Google Play reviews using continuation tokens until the target or end is reached."""
    collected_reviews = []
    continuation_token = None
    last_reported = 0

    try:
        while len(collected_reviews) < target_reviews:
            remaining = target_reviews - len(collected_reviews)
            batch_count = min(BATCH_SIZE, remaining)

            batch, continuation_token = reviews(
                app_package,
                lang="en",
                country="my",
                sort=Sort.NEWEST,
                count=batch_count,
                continuation_token=continuation_token,
            )

            if not batch:
                print("No more reviews were returned by Google Play.")
                break

            collected_reviews.extend(batch)

            if len(collected_reviews) - last_reported >= PROGRESS_INTERVAL:
                print(f"Collected {len(collected_reviews):,} reviews...")
                last_reported = len(collected_reviews)

            if continuation_token is None:
                print("Reached the final Google Play review page.")
                break
    except Exception as error:
        raise RuntimeError(f"Google Play scraping failed: {error}") from error

    if not collected_reviews:
        raise ValueError("No reviews were collected. Check internet access and package name.")

    print(f"Finished collection with {len(collected_reviews):,} reviews.")
    return collected_reviews


def convert_reviews_to_dataframe(review_records):
    """Keep only project-required fields and standardize column names."""
    rows = []
    for record in review_records:
        rows.append(
            {
                "review_id": record.get("reviewId"),
                "rating": record.get("score"),
                "review_date": record.get("at"),
                "app_version": record.get("reviewCreatedVersion"),
                "thumbs_up_count": record.get("thumbsUpCount"),
                "original_text": record.get("content"),
            }
        )

    dataframe = pd.DataFrame(rows, columns=RAW_COLUMNS)
    dataframe["review_date"] = pd.to_datetime(dataframe["review_date"], errors="coerce")
    dataframe["review_date"] = dataframe["review_date"].dt.strftime("%Y-%m-%d")
    return dataframe


def save_csv(dataframe, path):
    """Save a dataframe to CSV with a clear error if writing fails."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        dataframe.to_csv(output_path, index=False, encoding="utf-8-sig")
    except Exception as error:
        raise OSError(f"Could not save CSV file to {output_path}: {error}") from error

review_records = scrape_google_play_reviews(APP_PACKAGE, TARGET_REVIEWS)
raw_df = convert_reviews_to_dataframe(review_records)

if raw_df.empty:
    raise ValueError("Raw dataset is empty after conversion.")

save_csv(raw_df, "data/raw_data/shopee_my_reviews_raw.csv")
print("Raw dataset saved to: data/raw_data/shopee_my_reviews_raw.csv")
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()

Collected 600 reviews...
Collected 1,200 reviews...
Collected 1,800 reviews...
Collected 2,400 reviews...
Collected 3,000 reviews...
Collected 3,600 reviews...
Collected 4,200 reviews...
Collected 4,800 reviews...
Collected 5,400 reviews...
Collected 6,000 reviews...
Collected 6,600 reviews...
Collected 7,200 reviews...
Collected 7,800 reviews...
Collected 8,400 reviews...
Collected 9,000 reviews...
Collected 9,600 reviews...
Finished collection with 10,000 reviews.
Raw dataset saved to: data/raw_data/shopee_my_reviews_raw.csv
Raw dataset shape: (10000, 6)


,review_id,rating,review_date,app_version,thumbs_up_count,original_text
0,1ffac933-2b31-4730-8ab8-dc5dfdcb7e72,5,2026-07-01,3.76.27,0,ok
1,45a01044-dbe6-4e8e-bb29-c57ce58bedb4,5,2026-07-01,None,0,good
2,12c34620-7baa-4041-85a9-2fc1ff43a864,5,2026-07-01,3.76.27,3,I've had both wonderful & failed purchases exp...
3,f02f60c0-ee63-4526-9486-ab476587cc45,5,2026-07-01,3.76.27,0,Shoppee has helped many to get what they requi...
4,63567d2c-bb02-4395-862d-88c37ece06c2,5,2026-07-01,3.76.27,0,অংশ অংশ


## 4. Data Cleaning

This stage removes duplicate review IDs, duplicate review texts, null values, empty reviews, symbol-only reviews, and reviews shorter than three words.

In [4]:
def contains_letters_or_numbers(text):
    """Return True when a review has readable alphanumeric content."""
    return bool(re.search(r"[A-Za-z0-9]", str(text)))


def has_at_least_three_words(text):
    """Return True when a review has at least three word tokens."""
    return len(re.findall(r"\b\w+\b", str(text))) >= 3


def clean_raw_dataframe(dataframe):
    """Remove duplicates, missing values, empty reviews, symbol-only reviews, and very short reviews."""
    cleaned = dataframe.copy()
    initial_rows = len(cleaned)

    before = len(cleaned)
    cleaned = cleaned.drop_duplicates(subset="review_id")
    duplicate_id_removed = before - len(cleaned)

    before = len(cleaned)
    cleaned = cleaned.drop_duplicates(subset="original_text")
    duplicate_text_removed = before - len(cleaned)

    before = len(cleaned)
    cleaned = cleaned.dropna(subset=["review_id", "rating", "review_date", "original_text"])
    null_removed = before - len(cleaned)

    cleaned["original_text"] = cleaned["original_text"].astype(str).str.strip()

    before = len(cleaned)
    cleaned = cleaned[cleaned["original_text"] != ""]
    empty_removed = before - len(cleaned)

    before = len(cleaned)
    cleaned = cleaned[cleaned["original_text"].apply(contains_letters_or_numbers)]
    symbol_only_removed = before - len(cleaned)

    before = len(cleaned)
    cleaned = cleaned[cleaned["original_text"].apply(has_at_least_three_words)]
    short_removed = before - len(cleaned)

    cleaned = cleaned.reset_index(drop=True)

    stats = {
        "initial_rows": initial_rows,
        "duplicate_id_removed": duplicate_id_removed,
        "duplicate_text_removed": duplicate_text_removed,
        "duplicates_removed": duplicate_id_removed + duplicate_text_removed,
        "null_removed": null_removed,
        "empty_removed": empty_removed,
        "symbol_only_removed": symbol_only_removed,
        "short_removed": short_removed,
        "final_cleaned_rows": len(cleaned),
    }

    return cleaned, stats

clean_df, cleaning_stats = clean_raw_dataframe(raw_df)

print(f"Duplicate records removed: {cleaning_stats['duplicates_removed']:,}")
print(f"Null records removed: {cleaning_stats['null_removed']:,}")
print(f"Empty reviews removed: {cleaning_stats['empty_removed']:,}")
print(f"Symbol-only reviews removed: {cleaning_stats['symbol_only_removed']:,}")
print(f"Reviews shorter than three words removed: {cleaning_stats['short_removed']:,}")
print(f"Final number of cleaned reviews before NLP: {cleaning_stats['final_cleaned_rows']:,}")

if clean_df.empty:
    raise ValueError("No reviews remain after cleaning.")


Duplicate records removed: 2,291
Null records removed: 0
Empty reviews removed: 0
Symbol-only reviews removed: 82
Reviews shorter than three words removed: 1,232
Final number of cleaned reviews before NLP: 6,395


## 5. NLP Preprocessing

The `cleaned_text` column is created by applying lowercase conversion, URL/email/HTML removal, punctuation/number/emoji/special-character removal, whitespace normalization, tokenization, English and Malay stopword removal, and English lemmatization.

In [5]:
def remove_emojis(text):
    """Remove common emoji ranges from text."""
    emoji_pattern = re.compile(
        "["
        "\U0001F300-\U0001F5FF"
        "\U0001F600-\U0001F64F"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub(" ", str(text))


def download_nltk_resources():
    """Download only the NLTK resources required by this notebook."""
    resources = [
        ("corpora/stopwords", "stopwords"),
        ("corpora/wordnet", "wordnet"),
        ("corpora/omw-1.4", "omw-1.4"),
    ]

    for resource_path, package_name in resources:
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(package_name, quiet=True)


def get_stopword_set():
    """Create English and Malay stopword set while keeping important sentiment words."""
    english_stopwords = set(stopwords.words("english"))
    combined_stopwords = english_stopwords.union(MALAY_STOPWORDS)
    return combined_stopwords.difference(SENTIMENT_WORDS_TO_KEEP)


def preprocess_text(text, stopword_set, lemmatizer):
    """Apply lowercase, cleaning, tokenization, stopword removal, and English lemmatization."""
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w.%+-]+@[\w.-]+\.[A-Za-z]{2,}\b", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = html.unescape(text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)
    text = remove_emojis(text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [token for token in tokens if token not in stopword_set]
    tokens = [lemmatizer.lemmatize(token, wordnet.VERB) for token in tokens]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    return " ".join(tokens)


download_nltk_resources()
STOPWORD_SET = get_stopword_set()
LEMMATIZER = WordNetLemmatizer()


clean_df["cleaned_text"] = clean_df["original_text"].apply(
    lambda text: preprocess_text(text, STOPWORD_SET, LEMMATIZER)
)

before_empty_nlp = len(clean_df)
clean_df = clean_df[clean_df["cleaned_text"].str.strip() != ""].reset_index(drop=True)
empty_after_nlp_removed = before_empty_nlp - len(clean_df)

print(f"Rows removed after NLP because cleaned_text was empty: {empty_after_nlp_removed:,}")
print(f"Final cleaned reviews after NLP: {len(clean_df):,}")
clean_df[["original_text", "cleaned_text"]].head()

Rows removed after NLP because cleaned_text was empty: 0
Final cleaned reviews after NLP: 6,395


,original_text,cleaned_text
0,I've had both wonderful & failed purchases exp...,ive wonderful fail purchase experience shopee ...
1,Shoppee has helped many to get what they requi...,shoppee help many get require daily live witho...
2,"There's a bug issues lately, one for payment w...",there bug issue lately one payment lock last t...
3,fast to buy anithing,fast buy anithing
4,best app online shopping 🛍️ thanks shopee,best app online shop thank shopee


## 6. Exploratory Data Summary

This section prints summary statistics and word-frequency output only. No charts are used.

In [6]:
total_raw_reviews = len(raw_df)
total_cleaned_reviews = len(clean_df)

print("Dataset Summary")
print(f"Total raw reviews collected: {total_raw_reviews:,}")
print(f"Total cleaned reviews: {total_cleaned_reviews:,}")
print(f"Number of duplicates removed: {cleaning_stats['duplicates_removed']:,}")
print(f"Number of missing values removed: {cleaning_stats['null_removed']:,}")
print(f"Dataset dimensions: {clean_df.shape}")

print("\nColumn names:")
print(clean_df.columns.tolist())

rating_counts = clean_df["rating"].value_counts().sort_index()
average_rating = clean_df["rating"].mean()
print("\nFirst five records:")
display(clean_df.head())

print("\nRating Analysis")
print(f"Average rating: {average_rating:.2f}")
print("Rating counts:")
print(rating_counts)

original_word_count = clean_df["original_text"].apply(lambda text: len(str(text).split()))
cleaned_word_count = clean_df["cleaned_text"].apply(lambda text: len(str(text).split()))

print("\nReview Analysis")
print(f"Average original review length: {original_word_count.mean():.2f} words")
print(f"Average cleaned review length: {cleaned_word_count.mean():.2f} words")


Dataset Summary
Total raw reviews collected: 10,000
Total cleaned reviews: 6,395
Number of duplicates removed: 2,291
Number of missing values removed: 0
Dataset dimensions: (6395, 7)

Column names:
['review_id', 'rating', 'review_date', 'app_version', 'thumbs_up_count', 'original_text', 'cleaned_text']

First five records:


,review_id,rating,review_date,app_version,thumbs_up_count,original_text,cleaned_text
0,12c34620-7baa-4041-85a9-2fc1ff43a864,5,2026-07-01,3.76.27,3,I've had both wonderful & failed purchases exp...,ive wonderful fail purchase experience shopee ...
1,f02f60c0-ee63-4526-9486-ab476587cc45,5,2026-07-01,3.76.27,0,Shoppee has helped many to get what they requi...,shoppee help many get require daily live witho...
2,c4a59bfe-d2e9-44e9-a91b-d8efa563cb8b,3,2026-07-01,3.77.25,0,"There's a bug issues lately, one for payment w...",there bug issue lately one payment lock last t...
3,a8362235-614f-4051-8f0c-e60a87968e45,5,2026-07-01,3.77.25,0,fast to buy anithing,fast buy anithing
4,7a9a9fe1-d69f-4815-9a8b-c04ebb9b81cb,5,2026-07-01,3.77.25,0,best app online shopping 🛍️ thanks shopee,best app online shop thank shopee



Rating Analysis
Average rating: 3.54
Rating counts:
rating
1    1923
2     280
3     247
4     296
5    3649
Name: count, dtype: int64

Review Analysis
Average original review length: 19.97 words
Average cleaned review length: 12.44 words


## 7. Export Cleaned Dataset

In [7]:
save_csv(clean_df, "data/cleaned_data.csv")
print("Cleaned dataset saved to: data/cleaned_data.csv")
print(f"Final dataset columns: {clean_df.columns.tolist()}")

Cleaned dataset saved to: data/cleaned_data.csv
Final dataset columns: ['review_id', 'rating', 'review_date', 'app_version', 'thumbs_up_count', 'original_text', 'cleaned_text']
